#Dense RAG (Baseline)
___
Hệ thống sử dụng kiến trúc RAG cơ bản , kết hợp mô hình mã hóa vector ngữ nghĩa phẳng (Bi-Encoder vietnamese-sbert) và tìm kiếm khoảng cách Euclid bằng thư viện FAISS

In [ ]:
!pip install faiss-cpu sentence-transformers tqdm

In [4]:
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

def load_data():
    """Hàm nạp dữ liệu chunks thô và tập dữ liệu kiểm thử Ground Truth"""
    chunks_file = 'uet_rag_chunks_dataset.json'
    gt_file = 'retrieval_ground_truth.json'

    if not os.path.exists(chunks_file) or not os.path.exists(gt_file):
        raise FileNotFoundError(
            f"Vui lòng đảm bảo đã có file '{chunks_file}' và '{gt_file}' trong thư mục chạy code!"
        )

    with open(chunks_file, 'r', encoding='utf-8') as f:
        chunks_data = json.load(f)

    with open(gt_file, 'r', encoding='utf-8') as f:
        gt_dataset = json.load(f)

    return chunks_data, gt_dataset

def build_faiss_index(chunks_data, model_name='keepitreal/vietnamese-sbert'):
    """Hàm khởi tạo Embedding Model và nạp dữ liệu vector phẳng vào FAISS Index"""
    print(f"[*] Đang khởi tạo mô hình nhúng: {model_name}...")
    model = SentenceTransformer(model_name)

    print("[*] Đang tiến hành nhúng vector cho toàn bộ kho chunks dữ liệu...")
    all_texts = [chunk["text"] for chunk in chunks_data]
    doc_embeddings = model.encode(all_texts, show_progress_bar=True, convert_to_numpy=True)

    # Khởi tạo chỉ mục tìm kiếm khoảng cách L2
    dimension = doc_embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(doc_embeddings)

    print(f"[+] Khởi tạo Vector DB thành công với tổng số {index.ntotal} tài liệu.")
    return index, model

def run_benchmark(gt_dataset, chunks_data, faiss_index, embed_model, k_values=[1, 3, 5]):
    """Hàm chạy đánh giá tính toán chỉ số Hit Rate và MRR dựa trên Chunk ID"""
    print("\n[*] Bắt đầu quá trình chạy Benchmark Retrieval...")

    # Khởi tạo từ điển lưu trữ kết quả đếm cho từng mốc K
    results = {k: {"hits": 0, "mrr_score": 0.0} for k in k_values}
    total_queries = len(gt_dataset)
    max_k = max(k_values)

    for item in tqdm(gt_dataset, desc="Đang kiểm thử"):
        query = item["question"]
        target_chunk_id = item["ground_truth_chunk_id"] # Lấy ID chuẩn để đối chiếu

        # Chuyển đổi câu hỏi thành vector không gian
        query_embed = embed_model.encode([query], convert_to_numpy=True)

        # Truy xuất Top K khoảng cách ngắn nhất từ FAISS
        distances, indices = faiss_index.search(query_embed, k=max_k)

        # MẸO CỐT LÕI: Ánh xạ chỉ mục Index tìm được sang mã ID Chunk tương ứng
        retrieved_chunk_ids = [chunks_data[idx]["chunk_id"] for idx in indices[0]]

        # Tính toán chỉ số độc lập cho từng mốc K
        for k in k_values:
            sub_retrieved_ids = retrieved_chunk_ids[:k]

            # So sánh mã ID chuẩn có nằm trong Top-K bốc lên hay không
            if target_chunk_id in sub_retrieved_ids:
                results[k]["hits"] += 1

                # Xác định thứ hạng vị trí (1-indexed) của chunk đúng
                rank = sub_retrieved_ids.index(target_chunk_id) + 1
                results[k]["mrr_score"] += 1.0 / rank

    # Xuất báo cáo kết quả bảng chuẩn quy cách toán học
    print("\n" + "="*55)
    print(f"{'CẤU HÌNH K':<15}{'HIT RATE (Recall@K)':<25}{'MRR SCORE':<15}")
    print("-"*55)

    for k in k_values:
        hit_rate = (results[k]["hits"] / total_queries) * 100
        mrr = results[k]["mrr_score"] / total_queries
        print(f"Top-{k:<10}{hit_rate:<25.2f}%{mrr:<15.4f}")

    print("="*55)

if __name__ == "__main__":
    # Luồng thực thi tự động của hệ thống
    chunks, gt_data = load_data()
    faiss_idx, model = build_faiss_index(chunks, model_name='keepitreal/vietnamese-sbert')
    run_benchmark(gt_data, chunks, faiss_idx, model, k_values=[1, 3, 5])

[*] Đang khởi tạo mô hình nhúng: keepitreal/vietnamese-sbert...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[*] Đang tiến hành nhúng vector cho toàn bộ kho chunks dữ liệu...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

[+] Khởi tạo Vector DB thành công với tổng số 211 tài liệu.

[*] Bắt đầu quá trình chạy Benchmark Retrieval...


Đang kiểm thử: 100%|██████████| 103/103 [00:16<00:00,  6.27it/s]


CẤU HÌNH K     HIT RATE (Recall@K)      MRR SCORE      
-------------------------------------------------------
Top-1         51.46                    %0.5146         
Top-3         74.76                    %0.6197         
Top-5         83.50                    %0.6411         


#Reranked RAG (Advanced)
___
Hệ thống áp dụng quy trình truy xuất hai tầng (Two-stage Retrieval). Tầng một sử dụng tìm kiếm vector thô để đảm bảo độ phủ (Hit Rate), tầng hai tích hợp mô hình tương tác sâu Cross-Encoder (bge-reranker-base) để tái định vị thứ hạng tài liệu chuẩn xác lên vị trí Top-1 (tối ưu chỉ số MRR).

In [ ]:
!pip install faiss-cpu sentence-transformers tqdm

In [6]:
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm import tqdm

def load_data():
    """Hàm nạp dữ liệu chunks thô và tập dữ liệu kiểm thử Ground Truth"""
    chunks_file = 'uet_rag_chunks_dataset.json'
    gt_file = 'retrieval_ground_truth.json'

    if not os.path.exists(chunks_file) or not os.path.exists(gt_file):
        raise FileNotFoundError(
            f"Vui lòng đảm bảo đã upload file '{chunks_file}' và '{gt_file}' lên thư mục root của Colab!"
        )

    with open(chunks_file, 'r', encoding='utf-8') as f:
        chunks_data = json.load(f)

    with open(gt_file, 'r', encoding='utf-8') as f:
        gt_dataset = json.load(f)

    return chunks_data, gt_dataset

def build_faiss_index(chunks_data, model_name='keepitreal/vietnamese-sbert'):
    """Bước 1 của RAG: Khởi tạo Vector DB với Bi-Encoder"""
    print(f"[*] [TẦNG 1] Khởi tạo mô hình mã hóa nền: {model_name}...")
    model = SentenceTransformer(model_name)

    all_texts = [chunk["text"] for chunk in chunks_data]
    doc_embeddings = model.encode(all_texts, show_progress_bar=True, convert_to_numpy=True)

    dimension = doc_embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(doc_embeddings)

    print(f"[+] Lập chỉ mục thành công {index.ntotal} tài liệu phẳng.")
    return index, model, all_texts

def run_reranker_benchmark(gt_dataset, chunks_data, faiss_index, embed_model, all_texts, k_values=[1, 3, 5]):
    """Bước 2 của RAG: Tích hợp Cross-Encoder Reranker và tính toán chỉ số Benchmark"""

    # Khởi tạo mô hình Cross-Encoder Reranker từ HuggingFace để chạy trên Colab
    reranker_model_name = 'BAAI/bge-reranker-base'
    print(f"\n[*] [TẦNG 2] Khởi tạo mô hình chấm điểm chuyên sâu (Reranker): {reranker_model_name}...")
    reranker = CrossEncoder(reranker_model_name)

    print("\n[*] Bắt đầu chạy Benchmark Biến thể 2 (FAISS + Reranker)...")
    results = {k: {"hits": 0, "mrr_score": 0.0} for k in k_values}
    total_queries = len(gt_dataset)

    # Chiến lược: Thu thập Top-15 tài liệu thô từ FAISS để đảm bảo không bỏ sót, sau đó Rerank
    retrieve_top_n = 15

    for item in tqdm(gt_dataset, desc="Đang xử lý tập dữ liệu test"):
        query = item["question"]
        target_chunk_id = item["ground_truth_chunk_id"]

        # 1. Thực hiện truy xuất thô bằng vector phẳng
        query_embed = embed_model.encode([query], convert_to_numpy=True)
        distances, indices = faiss_index.search(query_embed, k=retrieve_top_n)

        # Lấy danh sách các ứng viên được bốc lên từ tầng 1
        candidate_indices = indices[0]
        candidate_chunks = [chunks_data[idx] for idx in candidate_indices]

        # 2. Tạo cặp dữ liệu (Câu hỏi, Tài liệu ứng viên) để đưa vào Cross-Encoder
        rerank_pairs = [[query, chunk["text"]] for chunk in candidate_chunks]

        # Tính toán điểm số tương tác sâu giữa câu hỏi và từng đoạn văn bản
        rerank_scores = reranker.predict(rerank_pairs)

        # Sắp xếp lại danh sách các ứng viên dựa trên điểm số Reranker từ cao xuống thấp
        sorted_indices = np.argsort(rerank_scores)[::-1]
        reranked_chunks = [candidate_chunks[idx] for idx in sorted_indices]

        # Lấy danh sách mã ID của các đoạn văn bản sau khi đã được tối ưu hóa thứ tự
        reranked_chunk_ids = [chunk["chunk_id"] for chunk in reranked_chunks]

        # 3. Tính toán lại chỉ số thực nghiệm
        for k in k_values:
            sub_retrieved_ids = reranked_chunk_ids[:k]

            if target_chunk_id in sub_retrieved_ids:
                results[k]["hits"] += 1
                rank = sub_retrieved_ids.index(target_chunk_id) + 1
                results[k]["mrr_score"] += 1.0 / rank

    # Xuất bảng kết quả chuẩn cấu thuật toán (Làm sạch lỗi chuỗi % cũ)
    print("\n" + "="*55)
    print(f"{'CẤU HÌNH K':<15}{'HIT RATE (Recall@K)':<25}{'MRR SCORE':<15}")
    print("-"*55)
    for k in k_values:
        hit_rate = (results[k]["hits"] / total_queries) * 100
        mrr = results[k]["mrr_score"] / total_queries
        print(f"Top-{k:<10}{hit_rate:<25.2f}%{mrr:<15.4f}")
    print("="*55)

if __name__ == "__main__":
    # Luồng chạy tự động
    chunks, gt_data = load_data()
    faiss_idx, bi_encoder, texts = build_faiss_index(chunks, model_name='keepitreal/vietnamese-sbert')
    run_reranker_benchmark(gt_data, chunks, faiss_idx, bi_encoder, texts, k_values=[1, 3, 5])

[*] [TẦNG 1] Khởi tạo mô hình mã hóa nền: keepitreal/vietnamese-sbert...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

[+] Lập chỉ mục thành công 211 tài liệu phẳng.

[*] [TẦNG 2] Khởi tạo mô hình chấm điểm chuyên sâu (Reranker): BAAI/bge-reranker-base...


...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]


[*] Bắt đầu chạy Benchmark Biến thể 2 (FAISS + Reranker)...


Đang xử lý tập dữ liệu test: 100%|██████████| 103/103 [24:59<00:00, 14.55s/it]


CẤU HÌNH K     HIT RATE (Recall@K)      MRR SCORE      
-------------------------------------------------------
Top-1         62.14                    %0.6214         
Top-3         83.50                    %0.7136         
Top-5         88.35                    %0.7252         
